# Сводка результатов всех экспериментов

Этот ноутбук собирает `result.csv` из всех экспериментов в единую таблицу  
и строит визуализации для сравнения.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

exp_root = Path().resolve()

## Сбор result.csv из всех экспериментов

In [ ]:
GROUP_COLORS = {
    "01_baselines":           "#4e9af1",
    "02_spectrogram_models":  "#f1a44e",
    "03_pretrained_frozen":   "#5cb85c",
    "04_pretrained_finetuned": "#9b59b6",
    "05_ensembles":           "#e74c3c",
    "06_phoneme_analysis":    "#95a5a6",
}

rows = []
for result_csv in sorted(exp_root.glob("**/result.csv")):
    group = result_csv.parts[-3] if len(result_csv.parts) >= 3 else "unknown"
    try:
        df = pd.read_csv(result_csv)
        df["group"] = group
        rows.append(df)
    except Exception as e:
        print(f"Ошибка чтения {result_csv}: {e}")

if rows:
    summary = pd.concat(rows, ignore_index=True)
    summary.to_csv(exp_root / "metrics_summary.csv", index=False, encoding="utf-8")
    print(f"Собрано {len(summary)} результатов")
    display(
        summary[
            ["experiment_id", "experiment_name", "group",
             "accuracy", "f1_macro", "f1_bad", "roc_auc",
             "threshold", "cv_f1_bad_std"]
        ].sort_values("f1_bad", ascending=False)
    )
else:
    print("Нет result.csv — запустите эксперименты")

## F1-bad по всем экспериментам

In [ ]:
if len(rows) == 0:
    print("Нет данных для графика")
else:
    df_plot = summary.dropna(subset=["f1_bad"]).sort_values("f1_bad", ascending=True)
    colors = [GROUP_COLORS.get(g, "grey") for g in df_plot["group"]]

    fig, ax = plt.subplots(figsize=(10, max(6, len(df_plot) * 0.4)))
    bars = ax.barh(df_plot["experiment_id"], df_plot["f1_bad"], color=colors)

    if "cv_f1_bad_std" in df_plot.columns:
        std_vals = df_plot["cv_f1_bad_std"].fillna(0)
        ax.errorbar(
            df_plot["f1_bad"], range(len(df_plot)),
            xerr=std_vals, fmt="none", ecolor="black",
            elinewidth=1.5, capsize=4
        )

    for bar, v in zip(bars, df_plot["f1_bad"]):
        ax.text(v + 0.004, bar.get_y() + bar.get_height()/2,
                f"{v:.3f}", va="center", fontsize=8)

    ax.set_xlabel("F1-bad", fontsize=12)
    ax.set_title("F1-bad по экспериментам (test test)", fontsize=13)
    ax.set_xlim(0, 1.0)

    patches = [mpatches.Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
    ax.legend(handles=patches, loc="lower right", fontsize=8)

    plt.tight_layout()
    plt.savefig(exp_root / "f1_bad_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("График сохранён в f1_bad_comparison.png")

## Сравнение по группам

In [ ]:
if len(rows) > 0 and len(summary) >= 4:
    group_order = list(GROUP_COLORS.keys())
    group_data = [
        summary[summary["group"] == g]["f1_bad"].dropna().values
        for g in group_order
        if g in summary["group"].values
    ]
    group_labels = [g for g in group_order if g in summary["group"].values]
    group_labels_short = [g.split("_", 1)[1] if "_" in g else g for g in group_labels]

    fig, ax = plt.subplots(figsize=(10, 5))
    bp = ax.boxplot(group_data, labels=group_labels_short, patch_artist=True)
    for patch, g in zip(bp["boxes"], group_labels):
        patch.set_facecolor(GROUP_COLORS.get(g, "grey"))
    ax.set_ylabel("F1-bad")
    ax.set_title("Распределение F1-bad по группам экспериментов")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(exp_root / "f1_bad_by_group.png", dpi=150, bbox_inches="tight")
    plt.show()

## Ключевые выводы

In [ ]:
if len(rows) > 0:
    best_exp = summary.loc[summary["f1_bad"].idxmax()]
    print(f"Лучший эксперимент: {best_exp['experiment_id']}")
    print(f"  Модель:    {best_exp['experiment_name']}")
    print(f"  F1-bad:    {best_exp['f1_bad']:.4f}")
    print(f"  F1-macro:  {best_exp['f1_macro']:.4f}")
    print(f"  Accuracy:  {best_exp['accuracy']:.4f}")
    print(f"  ROC-AUC:   {best_exp['roc_auc']:.4f}")
else:
    print("Запустите эксперименты и перезапустите этот ноутбук.")